# Go2: PID navigation + spiking skill selection

Run all cells for a headless moving-obstacle experiment and its plots. Set
`VIEWER = True` below to watch MuJoCo in real time. Select the project's
`.venv-win` Python environment. The existing `Go2_runner.ipynb` remains the
standalone walk demo.

The SNN selects **bypass left, bypass right, wait, or back off**. Joint torque
PID executes the selected skill, then normal destination tracking resumes.
Obstacle sensing currently uses ground-truth MuJoCo state. The trained SNN
imitates a geometric teacher; this is not reinforcement learning or a claim
of neuromorphic energy efficiency.


In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
assert (ROOT / "hybrid" / "system.py").exists(), "Open this notebook from the Go2-Learning project root."
print("Python:", sys.executable)
from hybrid.arena import scenario, Scenario
from hybrid.system import HybridSystem
from hybrid.run import save_result
from hybrid.snn import DEFAULT_MODEL
print("Saved SNN:", DEFAULT_MODEL)


## Choose the experiment

Try `crossing` to wait for a moving obstacle, `static` to bypass, or
`left_blocked` / `right_blocked` to constrain the direction.
`head_on` can trigger back-off. `blocked` deliberately has no usable passage.
Modes: `snn`, `rules` (teacher comparison), or `baseline` (no avoidance).
Each run starts a new episode; handoffs within an episode preserve physics.


In [ ]:
SCENARIO = "crossing"
MODE = "snn"
SEED = 0
VIEWER = False
REPLAN = False  # Try True with SCENARIO = "late_crossing" for experimental preemption.
OUTPUT = ROOT / "output" / "hybrid" / "notebook"

spec = scenario(SCENARIO, SEED)
# To edit the start, goal and obstacle motion, use this JSON instead:
# spec = Scenario.load(ROOT / "configs" / "hybrid_example.json")
print(json.dumps(spec.to_dict(), indent=2))


In [ ]:
result = HybridSystem(spec, mode=MODE, replanning=REPLAN).run(viewer=VIEWER)
save_result(result, OUTPUT)
print(json.dumps(result.summary, indent=2))


## Inspect the handoff and spikes

Blue trajectory segments track the destination; orange segments execute a
skill; purple segments settle before returning to destination tracking.
The raster shows 16 internal encoding steps of the first SNN decision.
These are not 16 sensor frames. Readout scores are not probabilities.


In [ ]:
from hybrid.plots import plot_run
from IPython.display import Image, display

figure = plot_run(OUTPUT)
display(Image(filename=str(figure)))
spikes = figure.with_name(figure.stem + "-spikes.png")
if MODE == "snn" and result.summary["interventions"] > 0 and spikes.exists():
    display(Image(filename=str(spikes)))
for event in result.events:
    print(f"{event['time']:6.1f}s  {event['source']:8s} -> {event['target']:8s}  "
          f"{event.get('skill', '')}  {event['reason']}")


## Compare the same scene

Set `RUN_COMPARISON = True` to execute all three controllers with identical
geometry. Failure is an experimental result: the direct baseline often
contacts the obstacle. Successful arrival needs 12 cm position accuracy,
low speed and a stopped gait for one second.


In [ ]:
RUN_COMPARISON = False
if RUN_COMPARISON:
    for mode in ("baseline", "rules", "snn"):
        comparison = HybridSystem(spec, mode=mode, replanning=REPLAN).run(viewer=False)
        save_result(comparison, OUTPUT / mode)
        summary = comparison.summary
        print(mode, "success:", summary["success"], "reason:", summary["reason"],
              "time:", round(summary["elapsed"], 2), "distance:", round(summary["final_distance"], 3))


## Repeatable batch runs and training

Run these from the project terminal:

```powershell
.\.venv-win\Scripts\python.exe -m hybrid.evaluate --seeds 100 101 102 --workers 3
.\.venv-win\Scripts\python.exe -m hybrid.plots --evaluation output/hybrid/evaluation
.\.venv-win\Scripts\python.exe -m hybrid.train_selector --output output/hybrid/retrained
.\.venv-win\Scripts\python.exe -m unittest discover -s tests -v
```

The shipped model is already trained. Retraining needs PyTorch from
`requirements-experiments.txt`; deployed inference needs only NumPy.
See `docs/hybrid_system.md` for architecture and limitations, and
`docs/hybrid_validation.md` for measured results. The physical robot,
camera/LiDAR sensing and arbitrary terrain are outside this prototype.
